In [2]:
# ============================================================
# Cell 0 — Parameters
# ============================================================
"""
Parameters from Table I of:
Gioannini & Rossetti, "Time-Domain Traveling Wave Model of Quantum Dot DFB Lasers"
IEEE J. Sel. Top. Quantum Electron., Vol. 17, No. 5, 2011

All values are in SI units unless noted.
"""

import numpy as np
import os

# ── Physical constants ────────────────────────────────────────────────────────
q        = 1.602176634e-19   # elementary charge (C)
hbar     = 1.054571817e-34   # reduced Planck constant (J·s)
kB       = 1.380649e-23      # Boltzmann constant (J/K)
c0       = 2.997924580e8     # speed of light in vacuum (m/s)
eps0     = 8.854187817e-12   # vacuum permittivity (F/m)
m0       = 9.1093837015e-31  # free electron mass (kg)
T        = 300.0             # room temperature (K)

# ── Material parameters ───────────────────────────────────────────────────────
h_w      = 5e-9              # QD layer height (m)
eta      = 3.3445            # effective refractive index (dimensionless)
N_l      = 8                 # number of QD layers
N_D      = 5.9e14            # QD surface density (m⁻²)  [5.9×10¹⁰ cm⁻² → ×10⁴]
N_groups = 51                # number of QD groups for inhomogeneous broadening

# State degeneracies: D_m for m = ES2, ES1, GS
D = {
    'ES2': 6,
    'ES1': 4,
    'GS':  2,
}

hbar_Gamma = 7e-3 * q       # homogeneous linewidth (J)  [7 meV → ×q]
Gamma      = hbar_Gamma / hbar  # dephasing rate (rad/s)

Delta_E    = 38e-3 * q      # FWHM of inhomogeneous broadening (J)  [38 meV]

# Dipole matrix elements A_m  (cm³·eV → m³·J)
# 1 cm³·eV = 1e-6 m³ × 1.602e-19 J  →  multiply by 1e-6 * q
A = {
    'ES1': 1.26e-20 * 1e-6 * q,   # m³·J
    'GS':  2.03e-20 * 1e-6 * q,   # m³·J
}

# SCH diffusion/transport times (s)
tau_c_e_W  = 1.2e-12        # electron transport time across SCH (s)
tau_c_h_W  = 23.2e-12       # hole transport time across SCH (s)

tau_e_e_W  = 0.0492e-12

# Electron relaxation times τ_c^{e,m} for m = ES2, ES1, GS (s)
tau_c_e = {
    'ES2': 3e-12,
    'ES1': 2e-12,
    'GS':  2e-12,
}

tau_e_e_im = {
    'ES2': 6.15e-12,
    'ES1': 7.99e-12,
    'GS':  10.63e-12,
}

# Interband recombination time in the WL (s)
tau_r_e_W  = 100e-12

tau_e_h_qd = 1.46E-12

# Spontaneous emission recombination times τ_Sp^m for m = ES2, ES1, GS (s)
tau_Sp = {
    'ES2': 2.8e-9,
    'ES1': 2.8e-9,
    'GS':  2.8e-9,
}

# Auger recombination times τ_Au^m for m = ES2, ES1, GS (s)
tau_Au = {
    'ES2': 110e-12,
    'ES1': 275e-12,
    'GS':  660e-12,
}

# Interband transition energies hbar*omega_im for i=(N+1)/2 (central group)
# m = ES2, ES1, GS  (eV → J)
hbar_omega = {
    'ES2': 1.098 * q,   # J
    'ES1': 1.042 * q,
    'GS':  0.959 * q,   # J
}

# Reference (Bragg) angular frequency — set to GS transition of central group
omega_0 = hbar_omega['GS'] / hbar   # rad/s

# Intrinsic waveguide losses (m⁻¹)  [1.5 cm⁻¹ → ×100]
alpha_i  = 1.5e2

# ── Device parameters ─────────────────────────────────────────────────────────
W        = 2.2e-6            # ridge equivalent width (m)
h_SCH    = 430e-9            # SCH height (m)
r0_sq    = 0.90              # power reflectivity at z=0  (HR facet)
rL_sq    = 0.00              # power reflectivity at z=L  (AR facet)
L        = 400e-6            # total cavity length (m)
k_DFB    = 40e2              # grating coupling coefficient (m⁻¹)  [40 cm⁻¹]

# ── Derived / assumed parameters ──────────────────────────────────────────────
# Internal quantum efficiency (50% per [7] in the paper)
eta_i    = 0.50

# Field confinement factor in QD layers (lateral × transverse)
# Not given in table; typical value for this ridge geometry
Gamma_xy = 0.03             # (dimensionless) — approximate; tune to match gain

# Optical confinement factor of the SCH
Gamma_xy_SCH = 0.3          # (dimensionless) — approximate

# ── QD group energies (Gaussian inhomogeneous distribution) ──────────────────
# Groups indexed i = 0 ... N_groups-1; central group i_c = (N_groups-1)//2
sigma    = Delta_E / (2 * np.sqrt(2 * np.log(2)))   # Gaussian sigma (J)
i_c      = (N_groups - 1) // 2                       # index of central group
i_arr    = np.arange(N_groups)                        # 0 … 50

# Energy offset of each group from central group (evenly spaced, ±3σ span)
# Total span set to 4×FWHM to capture >99% of distribution
E_span   = 4 * Delta_E                               # total span (J)
dE       = E_span / (N_groups - 1)                   # spacing between groups

# GS transition energy for each group
E_GS_i   = hbar_omega['GS'] + (i_arr - i_c) * dE    # (J), shape (N_groups,)

# ES1 and ES2 energies follow same offset as GS (rigid shift of whole spectrum)
E_ES1_i  = hbar_omega['ES1'] + (i_arr - i_c) * dE
E_ES2_i  = hbar_omega['ES2'] + (i_arr - i_c) * dE

# Gaussian weighting G_i (normalised so sum = 1)
G_i      = np.exp(-0.5 * ((i_arr - i_c) * dE / sigma) ** 2)
G_i     /= G_i.sum()

# ── Valence-band confined-state energies (hole states) ───────────────────────
# Five hole confined states (GS, ES1…ES4) equally separated by 12 meV
# (stated in Section III-A of the paper)
delta_h  = 12e-3 * q        # hole level separation (J)
# Hole state energies relative to WL (negative = below WL continuum)
# Labelled GS=0, ES1=1, ES2=2, ES3=3, ES4=4
n_hole_states = 5
E_hole   = np.array([-k * delta_h for k in range(n_hole_states)])  # (J)
D_hole   = np.array([2, 4, 6, 8, 10])  # degeneracy (assumed 2*(k+1))

# ── Collect everything into a single dictionary ───────────────────────────────
params = {
    # Physical constants
    'q':            q,
    'hbar':         hbar,
    'kB':           kB,
    'c0':           c0,
    'eps0':         eps0,
    'm0':           m0,
    'T':            T,

    # Material
    'h_w':          h_w,
    'eta':          eta,
    'N_l':          N_l,
    'N_D':          N_D,
    'N_groups':     N_groups,
    'D':            D,
    'hbar_Gamma':   hbar_Gamma,
    'Gamma':        Gamma,
    'Delta_E':      Delta_E,
    'A':            A,
    'tau_c_e_W':    tau_c_e_W,
    'tau_e_e_W':    tau_e_e_W,
    'tau_c_h_W':    tau_c_h_W,
    'tau_e_h_qd':   tau_e_h_qd,
    'tau_c_e':      tau_c_e,
    'tau_e_e_im':   tau_e_e_im,
    'tau_r_e_W':    tau_r_e_W,
    'tau_Sp':       tau_Sp,
    'tau_Au':       tau_Au,
    'hbar_omega':   hbar_omega,
    'omega_0':      omega_0,
    'alpha_i':      alpha_i,

    # Device
    'W':            W,
    'h_SCH':        h_SCH,
    'r0_sq':        r0_sq,
    'rL_sq':        rL_sq,
    'L':            L,
    'k_DFB':        k_DFB,

    # Derived / assumed
    'eta_i':        eta_i,

    'Gamma_xy':     Gamma_xy,
    'Gamma_xy_SCH': Gamma_xy_SCH,

    # QD group arrays (length N_groups)
    'i_arr':        i_arr,
    'G_i':          G_i,
    'E_GS_i':       E_GS_i,
    'E_ES1_i':      E_ES1_i,
    'E_ES2_i':      E_ES2_i,
    'dE':           dE,

    # Hole confined states
    'n_hole_states': n_hole_states,
    'E_hole':       E_hole,
    'D_hole':       D_hole,
    'delta_h':      delta_h,
}


if __name__ == '__main__':
    print("=== QD-DFB Laser Parameters (SI units) ===\n")
    for k, v in params.items():
        if isinstance(v, np.ndarray):
            print(f"  {k:20s}: array shape {v.shape}, "
                  f"range [{v.min():.4g}, {v.max():.4g}]")
        elif isinstance(v, dict):
            print(f"  {k:20s}: {v}")
        else:
            print(f"  {k:20s}: {v:.6g}" if isinstance(v, float) else
                  f"  {k:20s}: {v}")

=== QD-DFB Laser Parameters (SI units) ===

  q                   : 1.60218e-19
  hbar                : 1.05457e-34
  kB                  : 1.38065e-23
  c0                  : 2.99792e+08
  eps0                : 8.85419e-12
  m0                  : 9.10938e-31
  T                   : 300
  h_w                 : 5e-09
  eta                 : 3.3445
  N_l                 : 8
  N_D                 : 5.9e+14
  N_groups            : 51
  D                   : {'ES2': 6, 'ES1': 4, 'GS': 2}
  hbar_Gamma          : 1.12152e-21
  Gamma               : 1.06349e+13
  Delta_E             : 6.08827e-21
  A                   : {'ES1': 2.0187425588399996e-45, 'GS': 3.25241856702e-45}
  tau_c_e_W           : 1.2e-12
  tau_e_e_W           : 4.92e-14
  tau_c_h_W           : 2.32e-11
  tau_e_h_qd          : 1.46e-12
  tau_c_e             : {'ES2': 3e-12, 'ES1': 2e-12, 'GS': 2e-12}
  tau_e_e_im          : {'ES2': 6.15e-12, 'ES1': 7.99e-12, 'GS': 1.063e-11}
  tau_r_e_W           : 1e-10
  tau_Sp            

In [3]:
# ============================================================
# Cell 1 — Imports
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar

In [4]:

# ============================================================
# Cell 2 — Fixed arrays and constants
#           Assumes `params`, `i_arr`, `i_c`, `dE`, `sigma`
#           are already defined in your environment.
# ============================================================

N   = 51
n_m = 3   # [0] GS  [1] ES1  [2] ES2

# Inhomogeneous broadening distribution — shape (n_m, N)
G_i  = np.exp(-0.5 * ((i_arr - i_c) * dE / sigma) ** 2)
G_i /= G_i.sum()
G_i  = np.stack([G_i, G_i, G_i], axis=0)

# Degeneracy factors — shape (n_m, N)
D_m = np.array([2, 4, 6])[:, None] * np.ones((n_m, N))
A_m = np.array([params['A']['GS'], params['A']['ES1']])[:, None] * np.ones((2, N))

# Hole sub-band degeneracies and energies
D_m_h   = np.array([2, 4, 6, 8, 10], dtype=int)
delta_E = 12e-3 * 1.6e-19                           # [J]
E_h_m   = np.array([5, 4, 3, 2, 1]) * delta_E
E_h_WL  = 0.0

kBT = params['kB'] * params['T']                    # thermal energy [J]

# Auger and spontaneous-emission lifetimes — shape (n_m, N)
auger_lifetimes = np.zeros((n_m, N))
auger_lifetimes[0, :] = params['tau_Au']['GS']
auger_lifetimes[1, :] = params['tau_Au']['ES1']
auger_lifetimes[2, :] = params['tau_Au']['ES2']

sp_lifetimes = np.zeros((n_m, N))
sp_lifetimes[0, :] = params['tau_Sp']['GS']
sp_lifetimes[1, :] = params['tau_Sp']['ES1']
sp_lifetimes[2, :] = params['tau_Sp']['ES2']

omegahbar = np.array([params['E_GS_i']/params['hbar'],params['E_ES1_i']/params['hbar']],)

delta_E_sch_w   = 95.1 # UNCERTAIN - CLAUDE suggests 120 y
delta_E_h_sch_w = 154 # UNCERTAIN - CLAUDE suggests 150  z
delta_e_w_es2   = 36.9 # UNCERTAIN - CLAUDE suggests 52  x
delta_e_es2_es1 = 44 #CERTAIN
delta_e_es1_gs  = 71 #CERTAOM 


In [5]:
# ============================================================
# Cell 3 — Helper functions
#           These rely on kBT, E_h_m, E_h_WL, D_m_h, params
#           defined in Cell 2.
# ============================================================

def fun(EF, N_h_qd, params, m_h_w):
    """Charge-neutrality equation for holes (energies in Joules)."""
    wl_term = (m_h_w* 9.11E-31 * params['N_l'] * kBT
               / (np.pi * params['hbar']**2)
               ) * np.logaddexp(0.0, (E_h_WL - EF) / kBT)
    exponent      = (EF - E_h_m) / kBT
    confined_term = np.sum(params['N_l'] * params['N_D'] * D_m_h / (1.0 + np.exp(exponent)))
    return confined_term + wl_term - N_h_qd


def fun_eV(EF_eV, N_h_qd, params, m_h_w):
    """Wrapper for fun() that accepts EF in electronvolts."""
    return fun(EF_eV * 1.6e-19, N_h_qd, params, m_h_w)


def fermi(E, EF):
    """Fermi–Dirac occupation probability."""
    return 1.0 / (1.0 + np.exp((EF - E) / kBT))

def gain(omega, rho_e_im, rho_hmat, params):
    gain = (params['Gamma_xy']/params['h_w']*params['N_D']
            * np.sum(G_i[:2]*D_m[:2]*A_m*1j/params['hbar']/np.pi
                     / (params['Gamma']+1j*(omega-omegahbar))
                     * (rho_e_im[:2]+rho_hmat[:2]-1)))
    return gain

def gain_index(omega, rho_e_im, rho_hmat, params, index):

    if index == 0 or index ==1:

        gain = (params['Gamma_xy']/params['h_w']*params['N_D']
                * np.sum(G_i[index]*D_m[index]*A_m[index]*1j/params['hbar']/np.pi
                        / (params['Gamma']+1j*(omega-omegahbar[index]))
                        * (rho_e_im[index]+rho_hmat[index]-1)))
    if index == 2:
       gain = (params['Gamma_xy']/params['h_w']*params['N_D']
            * np.sum(G_i[0]*6*A_m[1]*1j/params['hbar']/np.pi
                     / (params['Gamma']+1j*(params['E_ES2_i']/params['hbar']))
                     * (rho_e_im[1]+rho_hmat[1]-1))) 
    return gain

In [10]:
def run_qd_simulation(t_end, initial_values, tstep, J=1E6, verbose=True,
                       threshold=0.01, params=None, delta_E_sch_w=95.1,
                       delta_E_h_sch_w=154, m_e_sch=0.063, m_e_w=0.03,
                       m_h_sch=0.5, m_h_w=0.45):
    if params is None:
        params = globals()['params']
    params = dict(params)
    rt_ev = 25.9
    delta_e_w_es2 = 286 - delta_E_sch_w - delta_E_h_sch_w

    DOS_sch   = 2*((2*np.pi*m_e_sch*9.11E-31*4.11E-21)/(6.626E-34)**2)**(3/2)
    DOS_w     = ((m_e_w*9.11E-31*4.11E-21)/(np.pi*params['hbar']**2))
    DOS_sch_h = 2*((2*np.pi*m_h_sch*9.11E-31*4.11E-21)/(6.626E-34)**2)**(3/2)
    DOS_w_h   = ((m_h_w*9.11E-31*4.11E-21)/(np.pi*params['hbar']**2))

    tau_e_e_W   = DOS_w*params['N_l']/DOS_sch/params['h_SCH']*np.exp(delta_E_sch_w/rt_ev)*params['tau_c_e_W']
    tau_e_h_qd  = 1/params['h_SCH']*DOS_w_h*params['N_l']/DOS_sch_h*np.exp(delta_E_h_sch_w/rt_ev)*params['tau_c_h_W']
    tau_e_e_ES1 = 4/6*np.exp(delta_e_es2_es1/rt_ev)*params['tau_c_e']['ES1']
    tau_e_e_ES2 = 6*params['N_D']/DOS_w*np.exp(delta_e_w_es2/rt_ev)*params['tau_c_e']['ES2']
    tau_e_e_GS  = 2/4*np.exp(delta_e_es1_gs/rt_ev)*params['tau_c_e']['GS']

    print(f'tau_e_e_W:   {tau_e_e_W}')
    print(f'tau_e_h_qd:  {tau_e_h_qd}')
    print(f'tau_e_e_ES1: {tau_e_e_ES1}')
    print(f'tau_e_e_ES2: {tau_e_e_ES2}')
    print(f'tau_e_e_GS:  {tau_e_e_GS}')
    # Unpack initial values
    n_e_sch = float(initial_values['n_e_sch'])
    n_e_w   = float(initial_values['n_e_w'])
    n_e_im  = initial_values['n_e_im'].copy().astype(float)
    n_h_sch = float(initial_values['n_h_sch'])
    n_hq_qd = float(initial_values['n_hq_qd'])

    n_steps   = int(np.round(t_end / tstep))
    milestone = max(1, n_steps // 10)

    for step in range(n_steps):
        if verbose and step % milestone == 0:
            pct = 100 * step / n_steps
            print(f"  {pct:5.1f}%  t = {step * tstep * 1e9:.3f} ns")

        rho_e_im = n_e_im / (params['N_l'] * params['N_D'] * G_i * D_m)

        EF_lo_eV = (E_h_m[0] - 100 * kBT) / 1.6e-19
        EF_hi_eV = (E_h_WL  + 100 * kBT)  / 1.6e-19
        sol      = root_scalar(fun_eV, args=(n_hq_qd, params, m_h_w),
                               bracket=[EF_lo_eV, EF_hi_eV], method='brentq')
        root_J   = sol.root * 1.6e-19

        rho_h_m   = fermi(E_h_m, root_J)
        rho_h_mat = np.zeros((n_e_im.shape[0], n_e_im.shape[1]))
        rho_h_mat[0, :] = rho_h_m[0]
        rho_h_mat[1, :] = rho_h_m[1]
        rho_h_mat[2, :] = rho_h_m[2]

        R_aug_im = n_e_im * rho_e_im * rho_h_mat / auger_lifetimes
        R_sp_im  = n_e_im * rho_h_mat / sp_lifetimes

        dn_e_sch = (params['eta_i']*J/params['q']
                    - n_e_sch/params['tau_c_e_W']
                    + n_e_w/tau_e_e_W)

        dn_e_w = (n_e_sch/params['tau_c_e_W']
                  - n_e_w/tau_e_e_W
                  - n_e_w/params['tau_r_e_W']
                  - np.sum(G_i[0,:]/params['tau_c_e']['ES2']*n_e_w*(1-rho_e_im[2,:]))
                  + np.sum(n_e_im[2,:]/tau_e_e_ES2))

        dn_i_e_es2 = (G_i[0,:]/params['tau_c_e']['ES2']*n_e_w*(1-rho_e_im[2,:])
                      - n_e_im[2,:]/tau_e_e_ES2
                      - R_sp_im[2,:] - R_aug_im[2,:]
                      - n_e_im[2,:]/params['tau_c_e']['ES1']*(1-rho_e_im[1,:])
                      + n_e_im[1,:]/tau_e_e_ES1*(1-rho_e_im[2,:]))

        dn_i_e_es1 = (n_e_im[2,:]/params['tau_c_e']['ES1']*(1-rho_e_im[1,:])
                      - n_e_im[1,:]/tau_e_e_ES1*(1-rho_e_im[2,:])
                      - R_aug_im[1,:]
                      - n_e_im[1,:]/params['tau_c_e']['GS']*(1-rho_e_im[0,:])
                      + n_e_im[0,:]/tau_e_e_GS*(1-rho_e_im[1,:])
                      - R_sp_im[1,:])

        dn_i_e_gs = (n_e_im[1,:]/params['tau_c_e']['GS']*(1-rho_e_im[0,:])
                     - n_e_im[0,:]/tau_e_e_GS*(1-rho_e_im[1,:])
                     - R_aug_im[0,:] - R_sp_im[0,:])

        dn_h_sch = (params['eta_i']*J/params['q']
                    - n_h_sch/params['tau_c_h_W']
                    + n_hq_qd/tau_e_h_qd)

        dn_h_qd  = (n_h_sch/params['tau_c_h_W']
                    - n_hq_qd/tau_e_h_qd
                    - np.sum(R_aug_im) - np.sum(R_sp_im)
                    - n_e_w / params['tau_r_e_W'])

        rel_changes = [
            abs(dn_e_sch) / (n_e_sch + 1e-20),
            abs(dn_e_w)   / (n_e_w   + 1e-20),
            abs(dn_h_sch) / (n_h_sch + 1e-20),
            abs(dn_h_qd)  / (n_hq_qd + 1e-20),
            np.max(abs(dn_i_e_gs)  / (n_e_im[0, :] + 1e-20)),
            np.max(abs(dn_i_e_es1) / (n_e_im[1, :] + 1e-20)),
            np.max(abs(dn_i_e_es2) / (n_e_im[2, :] + 1e-20)),
        ]

        n_e_sch      += dn_e_sch    * tstep
        n_e_w        += dn_e_w      * tstep
        n_e_im[0, :] += dn_i_e_gs  * tstep
        n_e_im[1, :] += dn_i_e_es1 * tstep
        n_e_im[2, :] += dn_i_e_es2 * tstep
        n_h_sch      += dn_h_sch    * tstep
        n_hq_qd      += dn_h_qd    * tstep

        if max(rel_changes) < threshold:
            if verbose:
                print(f"  Converged at step {step}  (t = {step * tstep * 1e9:.3f} ns)")
            break

    if verbose:
        print("  Done.")

    # Return only the final scalar/array state — no history
    final_state = {
        'n_e_sch': n_e_sch,
        'n_e_w':   n_e_w,
        'n_e_im':  n_e_im,        # shape (n_m_loc, N_loc)
        'n_h_sch': n_h_sch,
        'n_hq_qd': n_hq_qd,
    }
    t_final = step * tstep
    return t_final, final_state, step

In [ ]:
def plot_gain_and_carriers(final_state, len_omega=100, params=None,
                           wavelength_range_nm=None, m_h_w=0.45):
    if params is None:
        params = globals()['params']
    params = dict(params)

    if wavelength_range_nm is not None:
        wl_min_m = wavelength_range_nm[0] * 1e-9
        wl_max_m = wavelength_range_nm[1] * 1e-9
        omega_max = (2 * np.pi * params['c0']) / wl_min_m
        omega_min = (2 * np.pi * params['c0']) / wl_max_m
    else:
        omega_min = 1.35e15
        omega_max = 1.65e15

    omega = np.linspace(omega_min, omega_max, len_omega)

    n_e_im_final = final_state['n_e_im']       # shape (n_m_loc, N_loc)
    n_hq_qd_final = final_state['n_hq_qd']

    rho_e_im_final = n_e_im_final / (params['N_l'] * params['N_D'] * G_i * D_m)

    rho_e_gs_avg   = np.sum(G_i[0, :] * rho_e_im_final[0, :])
    rho_e_es1_avg  = np.sum(G_i[1, :] * rho_e_im_final[1, :])

    q_exact  = 1.602176634e-19
    EF_lo_eV = (E_h_m[0] - 15 * kBT) / q_exact
    EF_hi_eV = (E_h_WL   + 15 * kBT) / q_exact

    try:
        sol = root_scalar(fun_eV, args=(n_hq_qd_final, params, m_h_w),
                          bracket=[EF_lo_eV, EF_hi_eV], method='brentq')
        EF_h_final = sol.root * q_exact
    except ValueError as e:
        print(f"Root finding failed! n_hq_qd = {n_hq_qd_final:.4e}")
        raise e

    rho_h_m   = fermi(E_h_m, EF_h_final)
    rho_h_mat = np.zeros((2, 51))
    rho_h_mat[0, :] = rho_h_m[0]
    rho_h_mat[1, :] = rho_h_m[1]
    gain_spectrum = np.array([np.imag(gain(w, rho_e_im_final, rho_h_mat, params))
                           for w in omega])
    gain_cm       = gain_spectrum / 100
    wavelength_nm = (2 * np.pi * params['c0'] / omega) * 1e9

    rho_h_mat2 = np.zeros((3, 51))
    rho_h_mat2[0, :] = rho_h_m[0]
    rho_h_mat2[1, :] = rho_h_m[1]
    rho_h_mat2[2, :] = rho_h_m[2]
    aug_im = n_e_im_final * rho_e_im_final * rho_h_mat2 / auger_lifetimes
    R_sp_im  = n_e_im_final * rho_h_mat2 / sp_lifetimes

    gain_0 = np.array([np.imag(gain_index(w, rho_e_im_final, rho_h_mat, params, index=0))
                           for w in omega])/100
    gain_1 = np.array([np.imag(gain_index(w, rho_e_im_final, rho_h_mat, params, index=1))
                           for w in omega])/100
    gain_2 = np.array([np.imag(gain_index(w, rho_e_im_final, rho_h_mat, params, index=2))
                           for w in omega])/100

    refractive_index =  np.array([np.real(gain(w, rho_e_im_final, rho_h_mat, params)) for w in omega])/(2*params['eta'])

    """print(f"  ρ_e(GS)={rho_e_gs_avg:.4f}  ρ_e(ES1)={rho_e_es1_avg:.4f}")
    print(f"  ρ_h(GS)={rho_h_m[0]:.4f}  ρ_h(ES1)={rho_h_m[1]:.4f}")
    print(f"  Gain factor GS:  {rho_e_gs_avg + rho_h_m[0] - 1:.4f}")
    print(f"  Gain factor ES1: {rho_e_es1_avg + rho_h_m[1] - 1:.4f}")"""

    # Carrier totals from final state only
    n_e_qd = np.sum(n_e_im_final[0, :])

    total_electrons   = final_state['n_e_sch'] + final_state['n_e_w'] + n_e_qd
    total_holes       = final_state['n_h_sch'] + final_state['n_hq_qd']
    carrier_difference = total_electrons - total_holes

    

    return {
        'wavelength_nm':    wavelength_nm,
        'gain_cm_minus_1':  gain_cm,
        'total_electrons':  total_electrons,
        'total_holes':      total_holes,
        'carrier_difference': carrier_difference,
        'n_shift' : refractive_index,
        'gain_0' : gain_0,
        'gain_1' : gain_1,
        'gain_2' : gain_2,
        'R_aug_im' : aug_im,
        'R_sp_im' : R_sp_im
    }

In [8]:
# ============================================================
# Cell 5 — Define initial values and run
# ============================================================


def generate_custom_samples(total_points=500, high_density_ratio=0.6):
    """
    Generates a sample array for J from 0 to 18E8 with a higher density 
    of points concentrated between 4.5E6 and 18E6.
    
    Parameters:
    - total_points (int): Total number of samples in the final array.
    - high_density_ratio (float): Fraction of total points allocated to the target region.
    """
    # 1. Calculate how many points go into each of the 3 segments
    n_high = int(total_points * high_density_ratio)
    # Split the remaining points between the lower and upper outer zones
    n_remaining = total_points - n_high
    n_low = int(n_remaining * 0.25) # 0 to 4.5E6 is a narrow outer band
    n_upper = total_points - n_high - n_low
    
    # 2. Generate the segments
    # Use endpoint=False to prevent duplicate values at the boundary connections
    seg1 = np.linspace(0, 4.5e6, n_low, endpoint=False)
    seg2 = np.linspace(4.5e6, 18e6, n_high, endpoint=False)
    seg3 = np.linspace(18e6, 18e8, n_upper)
    
    # 3. Concatenate them together
    J = np.concatenate([seg1, seg2, seg3])
    return J


initial_values = {
    'n_e_sch' : 0.0,
    'n_e_w'   : 0.0,
    'n_e_im'  : np.zeros((n_m, N)),   # shape (n_m, N)
    'n_h_sch' : 0.0,
    'n_hq_qd' : 1e6,                 # initial hole population in QD reservoir
}

t_end = 20e-9    # [s]  — total simulation time  ← adjust as needed
tstep = 30e-15  # [s]  — timestep               ← adjust as needed
import numpy as np

# Example usage:
J_arr = generate_custom_samples(total_points=500, high_density_ratio=0.6)

N_arr = len(J_arr)
len_omega = 100
gain_array = np.zeros((N_arr,len_omega,))
N_array = np.zeros(N_arr)
rt_ev = 25.9

#### FIND change in the refractive index
epsilon_0 = 8.854E-12

def refractive_index_change(n_e_sch, n_e_w, n_e_im, n_h_sch, n_h_qd,
                             m_e=0.0465, m_h=0.455,
                             gamma_xy=0.8323233, gamma_xy_sch=0.6985092,
                             h_w=5E-9, h_sch=430E-9,dot = True):
    m_e_kg = m_e * 9.11E-31
    m_h_kg = m_h * 9.11E-31
    coeff = -params['q']**2 / (2*params['eta']*epsilon_0*omega_0**2)  # leading minus

    if dot:
        electron_term = (gamma_xy/(0.026*9.11E-31*h_w*params['N_l']))*(n_e_w+np.sum(n_e_im))
        hole_term = (gamma_xy/(0.4*9.11E-31*h_w*params['N_l']))*(n_h_qd)
    else:
        electron_term = gamma_xy_sch*n_e_sch/(0.064*9.11E-31**h_sch)
        hole_term =  gamma_xy_sch*n_h_sch/(0.5*9.11E-31**h_sch)
    return coeff*(electron_term + hole_term)

In [11]:
output_dir = 'gain_table'
os.makedirs(output_dir, exist_ok=True)

len_omega = 100

for J_val in J_arr:
    print(f'J= {J_val}')

    t_final, final_state, last_step = run_qd_simulation(
        t_end=30e-9,
        initial_values=initial_values,
        tstep=60e-15,
        threshold=0.05,
        J=J_val,
        verbose=True,
        params=None,
        m_h_w=0.4,
        m_e_w=0.026,
        delta_E_sch_w=75,
        delta_E_h_sch_w=140,
    )

    filename = f"J_{J_val}.npz"
    filepath = os.path.join(output_dir, filename)

    np.savez_compressed(
        filepath,
        t_final=np.array([t_final]),
        last_step=np.array([last_step]),
        **final_state,      # n_e_sch, n_e_w, n_e_im, n_h_sch, n_hq_qd
    )
    print(f"  Saved → {filepath}")

J= 0.0
tau_e_e_W:   2.869769446792076e-12
tau_e_h_qd:  4.695960593559146e-10
tau_e_e_ES1: 7.290147443730576e-12
tau_e_e_ES2: 5.910580830906274e-11
tau_e_e_GS:  1.5507328851608195e-11
    0.0%  t = 0.000 ns


KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

tau_e_e_W   = 2.869769446792076e-12
tau_e_h_qd  = 4.695960593559146e-10
tau_e_e_ES1 = 7.290147443730576e-12
tau_e_e_ES2 = 5.910580830906274e-11
tau_e_e_GS  = 1.5507328851608195e-11

output_dir = 'gain_table'
len_omega  = 100  # must match what was used during simulation

J_trimmed = J_arr[1:]
gain_array = np.zeros((len(J_trimmed), len_omega))
N_sweep_arr = np.zeros(len(J_trimmed))
n_alt = np.zeros(len(J_trimmed))

r_aug_gs_arr = np.zeros((len(J_trimmed),51))
r_aug_es1_arr = np.zeros((len(J_trimmed),51))
r_aug_es2_arr = np.zeros((len(J_trimmed),51))

r_sp_gs_arr = np.zeros((len(J_trimmed),51))
r_sp_es1_arr = np.zeros((len(J_trimmed),51))
r_sp_es2_arr = np.zeros((len(J_trimmed),51))

ref_index_arr = np.zeros(len(J_trimmed))
n_shift_arr = np.zeros((len(J_trimmed), len_omega))
gain_0_arr = np.zeros((len(J_trimmed), len_omega))
gain_1_arr = np.zeros((len(J_trimmed), len_omega))
gain_2_arr = np.zeros((len(J_trimmed), len_omega))

wavelength = None
for index, J_val in enumerate(J_trimmed):
    filename = f"J_{J_val}.npz"
    filepath = os.path.join(output_dir, filename)
    data = np.load(filepath)
    final_state = {
        'n_e_sch': float(data['n_e_sch']),
        'n_e_w':   float(data['n_e_w']),
        'n_e_im':  data['n_e_im'],          # shape (3, 51)
        'n_h_sch': float(data['n_h_sch']),
        'n_hq_qd': float(data['n_hq_qd']),
    }

    electron_number = np.sum(final_state['n_e_im'])+final_state['n_e_w']

    result_output = plot_gain_and_carriers(
        final_state,
        len_omega=len_omega,
    )

    gain_array[index] = result_output['gain_cm_minus_1']
    gain_0_arr[index] = result_output['gain_0']
    gain_1_arr[index] = result_output['gain_1']
    gain_2_arr[index] = result_output['gain_2']
    N_sweep_arr[index] = electron_number
    n_alt[index] = result_output['total_electrons'] 
    n_shift_arr[index] = result_output['n_shift']
    ref_index_arr[index] = refractive_index_change(final_state['n_e_sch'],final_state['n_e_w'], final_state['n_e_im'], final_state['n_h_sch'], final_state['n_hq_qd'],dot=True)

    r_aug_gs_arr[index] = result_output['R_aug_im'][0]
    r_aug_es1_arr[index] = result_output['R_aug_im'][1]
    r_aug_es2_arr[index] = result_output['R_aug_im'][2]

    r_sp_gs_arr[index] = result_output['R_sp_im'][0]
    r_sp_es1_arr[index] = result_output['R_sp_im'][1]
    r_sp_es2_arr[index] = result_output['R_sp_im'][2]

    if wavelength is None:
        wavelength = result_output['wavelength_nm']
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# EDIT this to match the hbar*Gamma used inside your gain() function
hbarGamma_J = 7e-3 * 1.602176634e-19  # 7 meV -> J
HC_J_NM = 1.986e-16  # J*nm

def lorentzian_lambda(lam_nm, lam0_nm, hbarGamma_J):
    E0 = HC_J_NM / lam0_nm
    E  = HC_J_NM / lam_nm
    g_E = (hbarGamma_J / np.pi) / ((E - E0)**2 + hbarGamma_J**2)
    jac = HC_J_NM / lam_nm**2
    return g_E * jac  # [1/nm]

def spectral_density(R_i, lambda_i_nm, lambda_grid_nm, hbarGamma_J):
    g = lorentzian_lambda(lambda_grid_nm[:, None], lambda_i_nm[None, :], hbarGamma_J)
    return (R_i[None, :] * g).sum(axis=1)

def write_picwave_spon_file(filepath, lambda_grid_nm, N_values_cm3, S_matrix, T_ref_C=25.0):
    n_lambda = len(lambda_grid_nm)
    n_N = len(N_values_cm3)
    lambda_min_um = lambda_grid_nm.min() / 1000.0
    lambda_max_um = lambda_grid_nm.max() / 1000.0

    order = np.argsort(N_values_cm3)
    N_sorted = np.array(N_values_cm3)[order]
    S_sorted = S_matrix[order, :]

    with open(filepath, 'w') as f:
        f.write("begin <nesponspectrum(6,2)>\n")
        f.write(f"{n_lambda} {lambda_min_um:.6f} {lambda_max_um:.6f} 1 {T_ref_C:.2f} {T_ref_C:.2f}\n")
        f.write(f"{n_N}\n")
        f.write(" ".join(f"{n:.6e}" for n in N_sorted) + "\n")
        for k in range(n_lambda):
            row = S_sorted[:, k]
            f.write(" ".join(f"{v:.6e}" for v in row) + "\n")
        f.write("end\n")

# ---- inputs already in your kernel ----
lambda_grid_nm = np.linspace(1050, 1400, 500)
lambda_GS_i  = HC_J_NM / E_GS_i
lambda_ES1_i = HC_J_NM / E_ES1_i
lambda_ES2_i = HC_J_NM / E_ES2_i

height_sch = 430e-7  # cm, SCH thickness — converts the sheet density N_sweep_arr
                      # into the volumetric N used as the lookup-table axis,
                      # matching your gain-table calibration convention

N_density_arr = N_sweep_arr*1E-4 / height_sch  # [cm^-3]

def build_matrix(r_arr, lambda_i_nm):
    S = np.zeros((len(N_sweep_arr), len(lambda_grid_nm)))
    for idx in range(len(N_sweep_arr)):
        S[idx] = spectral_density(r_arr[idx]*1E-4 / height3, lambda_i_nm, lambda_grid_nm, hbarGamma_J)
    return S

S_gs  = build_matrix(r_sp_gs_arr,  lambda_GS_i)
S_es1 = build_matrix(r_sp_es1_arr, lambda_ES1_i)
S_es2 = build_matrix(r_sp_es2_arr, lambda_ES2_i)

write_picwave_spon_file("spon_GS.txt",  lambda_grid_nm, N_density_arr, S_gs)
write_picwave_spon_file("spon_ES1.txt", lambda_grid_nm, N_density_arr, S_es1)
write_picwave_spon_file("spon_ES2.txt", lambda_grid_nm, N_density_arr, S_es2)
print("Wrote spon_GS.txt, spon_ES1.txt, spon_ES2.txt")

In [ ]:
# ── Write PICWAVE-compatible wide-band gain tables for gain_0 and gain_1 ──────
import numpy as np

# Define export targets and their respective arrays
export_targets = {
    'qd_gain0_TE.txt': gain_0_arr,
    'qd_gain1_TE.txt': gain_1_arr
}
import numpy as np

TEMPERATURE_C = 25.0
height = 40E-7 

# ── Convert to material gain [cm^-1] ─────────────────────────────────────────
# modal gain   = gain_array * conversion_factor          [cm^-1]
# material gain = modal gain / gamma_xy                  [cm^-1]
material_gain = gain_array * conversion_factor / gamma_xy   # shape (n_J, len_omega)

# ── Sort by carrier density (ascending) ──────────────────────────────────────
sort_idx      = np.argsort(N_sweep_arr)
N_sorted      = N_sweep_arr[sort_idx]*1E-4/height
gain_sorted   = material_gain[sort_idx]                     # (n_N, n_lambda)

sort_idx      = np.argsort(n_alt)
N_sorted      = n_alt[sort_idx]*1E-4/430E-7
gain_sorted   = material_gain[sort_idx]                     # (n_N, n_lambda)

# ── Sort by wavelength (ascending) ───────────────────────────────────────────
lam_sort_idx  = np.argsort(wavelength)
wavelength_sorted = wavelength[lam_sort_idx]
gain_sorted   = gain_sorted[:, lam_sort_idx]                # reorder λ axis to match

n_N           = len(N_sorted)
n_lambda      = len(wavelength_sorted)
lambda_min_um = wavelength_sorted.min() / 1000.0
lambda_max_um = wavelength_sorted.max() / 1000.0

print(f"N range  : {N_sorted[0]:.4e} – {N_sorted[-1]:.4e} cm^-3  ({n_N} points)")
print(f"λ range  : {lambda_min_um:.4f} – {lambda_max_um:.4f} um  ({n_lambda} points)")
print(f"Gain range: {gain_sorted.min():.1f} – {gain_sorted.max():.1f} cm^-1")
height2 = 430E-7 
N_sorted2 = N_sweep_arr[sort_idx]*1E-4/height2

for output_filename, raw_gain_data in export_targets.items():
    print(f"Processing {output_filename}...")
    
    # ── Convert to material gain [cm^-1] ─────────────────────────────────────
    # material gain = modal gain / gamma_xy
    material_gain = raw_gain_data * conversion_factor / gamma_xy
    
    # ── Sort by carrier density (ascending) ──────────────────────────────────
    # We reuse sort_idx and N_sorted from the previous cell to ensure consistency
    gain_sorted = material_gain[sort_idx]
    
    # ── Sort by wavelength (ascending) ───────────────────────────────────────
    # We reuse lam_sort_idx from the previous cell
    gain_sorted = gain_sorted[:, lam_sort_idx]
    
    print(f"  Gain range: {gain_sorted.min():.1f} – {gain_sorted.max():.1f} cm^-1")
    
    # ── Write file ────────────────────────────────────────────────────────────
    with open(output_filename, 'w') as f:
        f.write('begin <negainspectrum(1,0)>\n')
        f.write(
            f'{n_lambda} '
            f'{lambda_min_um:.6f} '
            f'{lambda_max_um:.6f} '
            f'1 '
            f'{TEMPERATURE_C:.1f} '
            f'{TEMPERATURE_C:.1f} '
            f'1\n'                   # poln = 1 (TE)
        )
        
        f.write(f'\n//T={TEMPERATURE_C:.0f} [C]\n')
        f.write(f'{n_N} //nN\n')
        
        # Carrier densities — one line, space-separated [cm^-3]
        f.write(' '.join(f'{n:.6e}' for n in N_sorted2) + '\n')
        
        # Gain block — one row per wavelength, columns = carrier densities
        for lam_idx in range(n_lambda):
            row = gain_sorted[:, lam_idx]
            f.write(' '.join(f'{g:.6e}' for g in row) + '\n')
            
        f.write('end\n')
        
    print(f"  Gain table written → {output_filename}\n")

print("Export complete.")
print("Add to your .mat file:")
print("  IMPORT_GAIN_SPECTRA_TE qd_gain0_TE_4.txt")
print("  IMPORT_GAIN_SPECTRA_TE qd_gain1_TE_4.txt")